# Exploratory Data Analysis 5.0

**Goal:** build pre-match rolling team-form features, export the enriched dataset, and evaluate an updated baseline logistic regression model.

## 1) Load and Chronologically Order Match Data

Read the processed match table, parse dates, and sort so that every engineered feature only uses past information.

In [1]:
# Load processed match-level data.
import pandas as pd

# Parse match dates and enforce time order for leak-free feature engineering.
matches_df = pd.read_csv('../data/processed/processed_matches.csv')
matches_df['date'] = pd.to_datetime(matches_df['date'])
matches_df = matches_df.sort_values('date').reset_index(drop=True)

# Quick sanity check of the loaded dataset.
matches_df.head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1
4,LOLTMNT05_171066,2026-01-09 17:09:20,16.01,LIT,HMBLE,P11 Esports,1


## 2) Engineer All-Time and Rolling Features

For each match, compute team features using only prior games:

- all-time form (career win-rate and games played up to that point)
- recent form (last 5 matches win-rate and games played)

This produces a per-match feature table aligned with the match outcome target.

In [2]:
# Number of most-recent matches used for rolling form.
window_size = 5

# Per-team stores: full rolling history and cumulative all-time totals.
team_history = {}
team_totals = {}

# Collect one engineered feature row per match.
feature_rows = []

for _, row in matches_df.iterrows():
    blue = row["blue_team"]
    red = row["red_team"]

    # Initialize tracking structures for unseen teams.
    if blue not in team_history:
        team_history[blue] = []
        team_totals[blue] = {"wins": 0, "games": 0}

    if red not in team_history:
        team_history[red] = []
        team_totals[red] = {"wins": 0, "games": 0}

    # Pre-match all-time stats.
    blue_games = team_totals[blue]["games"]
    red_games = team_totals[red]["games"]

    blue_wr = team_totals[blue]["wins"] / blue_games if blue_games > 0 else 0.5
    red_wr = team_totals[red]["wins"] / red_games if red_games > 0 else 0.5

    # Pre-match rolling stats from each team's latest `window_size` games.
    blue_recent = team_history[blue][-window_size:]
    red_recent = team_history[red][-window_size:]

    blue_games_5 = len(blue_recent)
    red_games_5 = len(red_recent)

    blue_wr_5 = sum(blue_recent) / blue_games_5 if blue_games_5 > 0 else 0.5
    red_wr_5 = sum(red_recent) / red_games_5 if red_games_5 > 0 else 0.5

    # Save engineered features alongside original match columns.
    feature_rows.append({
        **row,

        # ALL-TIME
        "blue_team_wr": blue_wr,
        "red_team_wr": red_wr,
        "blue_team_games": blue_games,
        "red_team_games": red_games,
        "wr_diff": blue_wr - red_wr,

        # ROLLING
        "blue_team_wr_5": blue_wr_5,
        "red_team_wr_5": red_wr_5,
        "blue_team_games_5": blue_games_5,
        "red_team_games_5": red_games_5,
        "wr_diff_5": blue_wr_5 - red_wr_5,
    })

    # Update team histories and totals after observing the match outcome.
    if row["blue_side_win"] == 1:
        team_totals[blue]["wins"] += 1
        team_history[blue].append(1)

        team_history[red].append(0)
    else:
        team_totals[red]["wins"] += 1
        team_history[blue].append(0)
        team_history[red].append(1)

    team_totals[blue]["games"] += 1
    team_totals[red]["games"] += 1

# Final engineered dataset used in downstream analysis and modeling.
combined_df = pd.DataFrame(feature_rows)
combined_df.head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win,blue_team_wr,red_team_wr,blue_team_games,red_team_games,wr_diff,blue_team_wr_5,red_team_wr_5,blue_team_games_5,red_team_games_5,wr_diff_5
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
4,LOLTMNT05_171066,2026-01-09 17:09:20,16.01,LIT,HMBLE,P11 Esports,1,0.0,1.0,1,1,-1.0,0.0,1.0,1,1,-1.0


## 3) Inspect Engineered Rolling Features

Run quick diagnostics to confirm the rolling features look reasonable before exporting and modeling.

In [3]:
# Preview core rolling features for the first few matches.
combined_df[[
    "blue_team_wr_5",
    "red_team_wr_5",
    "blue_team_games_5",
    "red_team_games_5",
    "wr_diff_5"
]].head(5)

,blue_team_wr_5,red_team_wr_5,blue_team_games_5,red_team_games_5,wr_diff_5
0,0.5,0.5,0,0,0.0
1,0.5,0.5,0,0,0.0
2,0.5,0.5,0,0,0.0
3,0.5,0.5,0,0,0.0
4,0.0,1.0,1,1,-1.0


In [4]:
# Summary statistics for rolling win-rate features.
combined_df[["blue_team_wr_5", "red_team_wr_5"]].describe()

,blue_team_wr_5,red_team_wr_5
count,5071.000000,5071.000000
mean,0.530020,0.496316
std,0.285963,0.282763
min,0.000000,0.000000
25%,0.400000,0.333333
50%,0.600000,0.500000
75%,0.800000,0.600000
max,1.000000,1.000000


In [5]:
# Inspect cold-start rows where the blue team has no rolling history yet.
combined_df[combined_df["blue_team_games_5"] == 0].head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win,blue_team_wr,red_team_wr,blue_team_games,red_team_games,wr_diff,blue_team_wr_5,red_team_wr_5,blue_team_games_5,red_team_games_5,wr_diff_5
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
8,LOLTMNT03_335584,2026-01-12 05:10:10,16.01,LCKC,Nongshim Esports Academy,DN SOOPers Challengers,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0


## 4) Export the Engineered Dataset

Persist the combined feature table so training scripts can reuse a stable, precomputed input file.

In [6]:
# Save engineered match features to the processed data directory.
combined_df.to_csv("../data/processed/final_feature_matches.csv", index=False)

## 5) Train and Evaluate a Baseline Model

Train a logistic regression baseline on pre-match features and evaluate predictive performance.

In [7]:
# Import train/test split, baseline classifier, and evaluation metrics.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Work on a copy to avoid accidental in-place edits.
model_df = combined_df.copy()

# Baseline feature set for this run.
features = [
    "wr_diff",
    "blue_team_games",
    "red_team_games"
]

# Build feature matrix and binary target.
X = model_df[features]
y = model_df["blue_side_win"]

# Hold out a test split for unbiased evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit logistic regression baseline.
model = LogisticRegression()
model.fit(X_train, y_train)

# Generate predictions and print evaluation metrics.
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6344827586206897
Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.47      0.55       481
           1       0.62      0.79      0.69       534

    accuracy                           0.63      1015
   macro avg       0.64      0.63      0.62      1015
weighted avg       0.64      0.63      0.62      1015



## 6) Interpret Model Coefficients

Inspect learned coefficients to understand each feature's direction and relative influence.

In [8]:
# Pair each feature with its learned logistic regression coefficient.
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})

print(coefficients)

           feature  coefficient
0          wr_diff     2.252755
1  blue_team_games     0.014774
2   red_team_games    -0.010807
